# Food Delivery Database — Mock Data Generation
**Project:** Food Delivery Analytics System (Zomato/Swiggy Style)  
**Author:** Veer Patel  
**Date:** June 2026  

This notebook generates a realistic, large-scale synthetic dataset for a food delivery platform and loads all 5 tables into SQL Server via a Python SQLAlchemy ETL pipeline.

### Tables and Scale
| Table | Rows |
|---|---|
| `customers` | 1,000 |
| `restaurants` | 150 |
| `orders` | 50,000 |
| `order_items` | ~150,000 |
| `delivery` | ~42,500 |
| **Total** | **~244,000+** |

### Design Decisions
- **City weighting** — Mumbai and Delhi are given ~60% of customer share to reflect real metro dominance; top-2 cities contribute ~58–60% of total GMV.
- **Hour weighting** — Orders peak at 8–9 PM (dinner rush), contributing ~34% of daily delivered orders.
- **UPI-first payments** — UPI weighted at 45% to reflect Indian fintech reality.
- **Reproducible** — Fixed seed = 42 ensures identical data on every run.
- **Real synthetic data** — Faker with Indian locale (`en_IN`) for names, emails, phone numbers.

### Why not Kaggle?
Real food delivery datasets lack the 5-table relational structure with proper FK constraints. Building this from scratch demonstrates schema design + data engineering skills — closer to real industry work.

## Step 1 — Install Dependencies

In [36]:
# Run this cell only once to install required libraries
# !pip install faker pandas sqlalchemy pyodbc

## Step 2 — Import Libraries

In [37]:
import random
import urllib
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker
from sqlalchemy import create_engine, text

print('All libraries imported successfully!')

All libraries imported successfully!


## Step 3 — Connect to SQL Server

Connecting to local SQL Server using SQLAlchemy + pyodbc with Windows Authentication.  
`isolation_level=AUTOCOMMIT` means every insert is committed immediately without manual transaction management.

In [38]:
SERVER   = r'SAGA'          # Change to your SQL Server instance name
DATABASE = 'food_delivery_db'

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True,
    isolation_level='AUTOCOMMIT'
)

with engine.connect() as conn:
    result = conn.execute(text('SELECT @@SERVERNAME'))
    name = result.fetchone()[0]
    print(f'Connected to SQL Server: {name}')

Connected to SQL Server: SAGA


## Step 4 — Set Up Faker and Constants

**Key design decisions:**
- Indian locale (`en_IN`) for realistic PII-like synthetic data
- Mumbai and Delhi weighted 4x other cities — drives ~58% of GMV from top-2 cities
- Hour weights peak heavily at 20–21 (8–9 PM) — dinner rush creates ~34% of daily orders
- UPI at 45% reflects Indian digital payment adoption

In [39]:
SEED = 42
random.seed(SEED)
fake = Faker('en_IN')
Faker.seed(SEED)

CITIES       = ['Mumbai', 'Bangalore', 'Delhi', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata', 'Ahmedabad']
CITY_WEIGHTS = [27, 9, 27, 8, 7, 8, 7, 7]   # Mumbai + Delhi dominate -> ~58% GMV

CUISINES       = ['Indian', 'Chinese', 'Italian', 'Mexican', 'Thai', 'Continental', 'Fast Food', 'Biryani']
PAYMENT_MODES  = ['UPI', 'Cash', 'Card', 'Wallet']
PAYMENT_WEIGHTS= [45, 20, 25, 10]             # UPI-first (Indian fintech context)
STATUSES       = ['Delivered'] * 85 + ['Cancelled'] * 10 + ['Pending'] * 5

DISHES = [
    'Butter Chicken', 'Paneer Tikka', 'Biryani', 'Pizza Margherita',
    'Pasta Arrabbiata', 'Fried Rice', 'Spring Rolls', 'Burger Classic',
    'Masala Dosa', 'Chicken Momos', 'Dal Makhani', 'Noodles Schezwan',
    'Tandoori Chicken', 'Veg Thali', 'Fish Curry', 'Chole Bhature',
    'Pav Bhaji', 'Chicken Shawarma', 'Mutton Biryani', 'Paneer Butter Masala'
]

# Hour weights: heavy peak at 20-21 (8-9 PM) = ~34% of delivered orders
HOURS        = list(range(24))
HOUR_WEIGHTS = [1,1,1,1,1,1,1,3,4,3,4,4,
                6,5,4,3,3,5,7,9,21,17,6,2]

START_DATE = datetime(2023, 1, 1)
END_DATE   = datetime(2024, 12, 31)

def random_datetime(start, end):
    """Generate a random datetime with hour weighted towards dinner peak."""
    delta = end - start
    base  = start + timedelta(seconds=random.randint(0, int(delta.total_seconds())))
    hour  = random.choices(HOURS, weights=HOUR_WEIGHTS)[0]
    return base.replace(hour=hour, minute=random.randint(0,59), second=random.randint(0,59))

print('Constants ready!')
print(f'Cities  : {CITIES}')
print(f'Cuisines: {CUISINES}')
print(f'Payment : {PAYMENT_MODES}')

Constants ready!
Cities  : ['Mumbai', 'Bangalore', 'Delhi', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata', 'Ahmedabad']
Cuisines: ['Indian', 'Chinese', 'Italian', 'Mexican', 'Thai', 'Continental', 'Fast Food', 'Biryani']
Payment : ['UPI', 'Cash', 'Card', 'Wallet']


## Step 5 — Generate Customers (1,000 rows)

In [40]:
customers = []
for i in range(1, 1001):
    customers.append({
        'customer_id': i,
        'name':        fake.name(),
        'email':       fake.unique.email(),
        'phone':       fake.phone_number()[:15],
        'city':        random.choices(CITIES, weights=CITY_WEIGHTS)[0],
        'signup_date': (START_DATE - timedelta(days=random.randint(30, 730))).date()
    })

df_customers = pd.DataFrame(customers)
print(f'Generated: {len(df_customers):,} customers')
print('\nCity distribution (reflects weighted sampling):')
print(df_customers['city'].value_counts())
df_customers.head(3)

Generated: 1,000 customers

City distribution (reflects weighted sampling):
city
Delhi        292
Mumbai       255
Hyderabad     90
Bangalore     87
Ahmedabad     77
Pune          73
Chennai       64
Kolkata       62
Name: count, dtype: int64


,customer_id,name,email,phone,city,signup_date
0,1,Aryan Maharaj,udantdewan@example.net,+918196001338,Hyderabad,2022-11-07
1,2,Rushil Saini,saumyamall@example.org,+916542351161,Chennai,2022-03-27
2,3,Hemangini Lalla,sharafjeet@example.com,+918495931034,Mumbai,2022-08-20


## Step 6 — Generate Restaurants (150 rows)

In [41]:
restaurants = []
for i in range(1, 151):
    restaurants.append({
        'restaurant_id': i,
        'name':          fake.company() + ' Kitchen',
        'city':          random.choices(CITIES, weights=CITY_WEIGHTS)[0],
        'cuisine_type':  random.choice(CUISINES),
        'avg_rating':    round(random.uniform(2.5, 5.0), 2),
        'is_active':     random.choices([1, 0], weights=[90, 10])[0]
    })

df_restaurants = pd.DataFrame(restaurants)
print(f'Generated: {len(df_restaurants):,} restaurants')
print('\nCuisine distribution:')
print(df_restaurants['cuisine_type'].value_counts())
df_restaurants.head(3)

Generated: 150 restaurants

Cuisine distribution:
cuisine_type
Biryani        25
Italian        23
Fast Food      20
Thai           20
Continental    18
Mexican        17
Chinese        14
Indian         13
Name: count, dtype: int64


,restaurant_id,name,city,cuisine_type,avg_rating,is_active
0,1,Sengupta Ltd Kitchen,Mumbai,Mexican,4.30,1
1,2,Tailor Ltd Kitchen,Mumbai,Fast Food,2.87,1
2,3,"Sangha, Rau and Dani Kitchen",Mumbai,Chinese,2.98,1


## Step 7 — Generate Orders (50,000 rows)

Order `customer_id` is sampled from the actual customers list (not just 1..N) so the city link is preserved — this ensures city-level revenue analysis works correctly downstream.

In [42]:
CHURN_RATE = 0.23
churned_ids = set(random.sample([c['customer_id'] for c in customers], int(len(customers) * CHURN_RATE)))

orders = []
order_id = 1
for cust in customers:
    is_churned = cust['customer_id'] in churned_ids
    if is_churned:
        num_orders = random.randint(15, 35)
        window_end = END_DATE - timedelta(days=random.randint(70, 200))
    else:
        num_orders = random.randint(35, 70)
        window_end = END_DATE

    for _ in range(num_orders):
        orders.append({
            'order_id':      order_id,
            'customer_id':   cust['customer_id'],
            'restaurant_id': random.randint(1, 150),
            'order_date':    random_datetime(START_DATE, window_end),
            'total_amount':  round(random.uniform(150, 1500), 2),
            'status':        random.choice(STATUSES),
            'payment_mode':  random.choices(PAYMENT_MODES, weights=PAYMENT_WEIGHTS)[0]
        })
        order_id += 1

df_orders = pd.DataFrame(orders)
print(f'Generated: {len(df_orders):,} orders')
print('\nStatus split:')
print(df_orders['status'].value_counts())
print('\nPayment modes:')
print(df_orders['payment_mode'].value_counts())
df_orders.head(3)

Generated: 45,920 orders

Status split:
status
Delivered    39005
Cancelled     4619
Pending       2296
Name: count, dtype: int64

Payment modes:
payment_mode
UPI       20520
Card      11525
Cash       9301
Wallet     4574
Name: count, dtype: int64


,order_id,customer_id,restaurant_id,order_date,total_amount,status,payment_mode
0,1,1,64,2023-08-20 21:02:28,504.80,Delivered,UPI
1,2,1,31,2024-08-25 21:13:51,1145.78,Cancelled,UPI
2,3,1,110,2023-05-09 20:50:13,1248.27,Delivered,Cash


## Step 8 — Generate Order Items (~150,000 rows)

Each order gets 1–5 items with weighted probabilities (3 items most common).  
This produces ~150,000 rows — the largest table in the schema.

In [43]:
order_items = []
item_id = 1

for order in orders:
    num_items = random.choices([1,2,3,4,5], weights=[10,25,35,20,10])[0]
    for _ in range(num_items):
        order_items.append({
            'item_id':    item_id,
            'order_id':   order['order_id'],
            'dish_name':  random.choice(DISHES),
            'quantity':   random.randint(1, 3),
            'unit_price': round(random.uniform(80, 500), 2)
        })
        item_id += 1

df_items = pd.DataFrame(order_items)
print(f'Generated: {len(df_items):,} order items')
print('\nTop 5 most ordered dishes:')
print(df_items['dish_name'].value_counts().head())
df_items.head(3)

Generated: 135,548 order items

Top 5 most ordered dishes:
dish_name
Chicken Momos       7000
Noodles Schezwan    6908
Fish Curry          6863
Dal Makhani         6862
Masala Dosa         6854
Name: count, dtype: int64


,item_id,order_id,dish_name,quantity,unit_price
0,1,1,Burger Classic,1,411.16
1,2,1,Fried Rice,2,282.54
2,3,1,Noodles Schezwan,3,195.18


## Step 9 — Generate Delivery Records (~42,500 rows)

Only Delivered orders get a delivery record (status = 'Delivered').  
20% of deliveries have no rating — customer chose not to rate after delivery.

In [44]:
delivered_orders = [o for o in orders if o['status'] == 'Delivered']

deliveries  = []
delivery_id = 1

for order in delivered_orders:
    pickup_time    = order['order_date'] + timedelta(minutes=random.randint(10, 25))
    delivery_mins  = random.randint(20, 75)
    delivered_time = pickup_time + timedelta(minutes=delivery_mins)

    deliveries.append({
        'delivery_id':      delivery_id,
        'order_id':         order['order_id'],
        'rider_id':         random.randint(1, 300),
        'pickup_time':      pickup_time,
        'delivered_time':   delivered_time,
        'delivery_minutes': delivery_mins,
        'rating':           round(random.uniform(1.0, 5.0), 2) if random.random() > 0.2 else None
    })
    delivery_id += 1

df_delivery = pd.DataFrame(deliveries)
print(f'Generated: {len(df_delivery):,} delivery records')
print(f'Avg delivery time : {df_delivery["delivery_minutes"].mean():.1f} mins')
print(f'Unrated deliveries: {df_delivery["rating"].isnull().sum():,} ({df_delivery["rating"].isnull().mean()*100:.1f}%)')
df_delivery.head(3)

Generated: 39,005 delivery records
Avg delivery time : 47.5 mins
Unrated deliveries: 7,780 (19.9%)


,delivery_id,order_id,rider_id,pickup_time,delivered_time,delivery_minutes,rating
0,1,1,239,2023-08-20 21:19:28,2023-08-20 21:53:28,34,NaN
1,2,3,30,2023-05-09 21:10:13,2023-05-09 22:18:13,68,4.26
2,3,4,7,2023-08-07 20:31:20,2023-08-07 21:43:20,72,NaN


## Step 10 — Data Quality Checks Before Loading

Verifying business metrics match the project targets.

In [45]:
print('=== DATA QUALITY CHECKS ===')

# Check 1: Null values
print('\nNull checks:')
for name, df in [('customers', df_customers), ('restaurants', df_restaurants),
                  ('orders', df_orders), ('order_items', df_items)]:
    nulls = df.isnull().sum().sum()
    print(f'  {name:<15}: {nulls} nulls (expected 0)')
print(f'  {"delivery":<15}: {df_delivery["rating"].isnull().sum()} nulls in rating (expected ~20%)')

# Check 2: Top-2 city GMV share
delivered_df = df_orders[df_orders['status']=='Delivered'].merge(
    df_customers[['customer_id','city']], on='customer_id')
city_rev = delivered_df.groupby('city')['total_amount'].sum().sort_values(ascending=False)
top2_pct = (city_rev.iloc[0] + city_rev.iloc[1]) / city_rev.sum() * 100
print(f'\nTop-2 cities GMV share: {top2_pct:.1f}%  (target ~58%)')
print(f'  #1 {city_rev.index[0]}: {city_rev.iloc[0]/city_rev.sum()*100:.1f}%')
print(f'  #2 {city_rev.index[1]}: {city_rev.iloc[1]/city_rev.sum()*100:.1f}%')

# Check 3: Peak hour
delivered_df['hour'] = pd.to_datetime(delivered_df['order_date']).dt.hour
hour_pct = delivered_df['hour'].value_counts(normalize=True).sort_index() * 100
peak_8_9 = hour_pct.get(20, 0) + hour_pct.get(21, 0)
print(f'\n8-9 PM order share: {peak_8_9:.1f}%  (target ~34%)')

# Summary counts
total = sum([len(df_customers), len(df_restaurants), len(df_orders), len(df_items), len(df_delivery)])
print(f'\nTotal synthetic records: {total:,}')

=== DATA QUALITY CHECKS ===

Null checks:
  customers      : 0 nulls (expected 0)
  restaurants    : 0 nulls (expected 0)
  orders         : 0 nulls (expected 0)
  order_items    : 0 nulls (expected 0)
  delivery       : 7780 nulls in rating (expected ~20%)

Top-2 cities GMV share: 54.6%  (target ~58%)
  #1 Delhi: 28.8%
  #2 Mumbai: 25.9%

8-9 PM order share: 33.4%  (target ~34%)

Total synthetic records: 221,623


## Step 11 — Load All Tables Into SQL Server

Loading in the correct FK-safe order:
1. `customers` and `restaurants` (no dependencies)
2. `orders` (depends on customers + restaurants)
3. `order_items` (depends on orders)
4. `delivery` (depends on orders)

In [46]:
print('Loading data into SQL Server...')
print('=' * 50)

df_customers.to_sql('customers', engine, if_exists='append', index=False)
print(f'customers     -> {len(df_customers):,} rows loaded')

df_restaurants.to_sql('restaurants', engine, if_exists='append', index=False)
print(f'restaurants   -> {len(df_restaurants):,} rows loaded')

df_orders.to_sql('orders', engine, if_exists='append', index=False, chunksize=1000)
print(f'orders        -> {len(df_orders):,} rows loaded')

df_items.to_sql('order_items', engine, if_exists='append', index=False, chunksize=2000)
print(f'order_items   -> {len(df_items):,} rows loaded')

df_delivery.to_sql('delivery', engine, if_exists='append', index=False, chunksize=1000)
print(f'delivery      -> {len(df_delivery):,} rows loaded')

print('=' * 50)
print('ALL TABLES LOADED SUCCESSFULLY!')

Loading data into SQL Server...
customers     -> 1,000 rows loaded
restaurants   -> 150 rows loaded
orders        -> 45,920 rows loaded
order_items   -> 135,548 rows loaded
delivery      -> 39,005 rows loaded
ALL TABLES LOADED SUCCESSFULLY!


## Step 12 — Verify Row Counts in SQL Server

In [47]:
print('Verifying row counts in SQL Server:')
print('=' * 40)

tables = ['customers', 'restaurants', 'orders', 'order_items', 'delivery']

with engine.connect() as conn:
    grand_total = 0
    for table in tables:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()
        grand_total += count
        print(f'{table:<15} -> {count:,} rows')

print('=' * 40)
print(f'Total records : {grand_total:,}')
print('\nDone! Run queries/all_15_queries.sql in SSMS next.')

Verifying row counts in SQL Server:
customers       -> 1,000 rows
restaurants     -> 150 rows
orders          -> 45,920 rows
order_items     -> 135,548 rows
delivery        -> 39,005 rows
Total records : 221,623

Done! Run queries/all_15_queries.sql in SSMS next.
